<a href="https://colab.research.google.com/github/SebastianSoto17/Computational-Intelligence/blob/main/Participacion_19_marzo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Participación 19 de marzo

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

In [ ]:
from sklearn.datasets import fetch_california_housing
data = fetch_california_housing()

X, y = data.data, data.target

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print(f"Train 80%: {X_train.shape[0]} muestras ")
print(f"Validation 10%:   {X_val.shape[0]} muestras ")
print(f"Test 10%:  {X_test.shape[0]} muestras ")

Train 80%: 16512 muestras 
Validation 10%:   2064 muestras 
Test 10%:  2064 muestras 


In [ ]:
import torch
import torch.nn as nn

In [ ]:
class SimplePreprocessor(nn.Module):

    def __init__(self, feature_names, excluded=("Latitude", "Longitude"), iqr_k=1.5, eps=0.000001):
        super().__init__()
        self.feature_names = list(feature_names)
        self.iqr_k = iqr_k
        self.eps = eps

        proc_idx = [i for i, n in enumerate(self.feature_names) if n not in excluded]
        raw_idx = [i for i, n in enumerate(self.feature_names) if n in excluded]

        self.register_buffer("proc_idx", torch.tensor(proc_idx, dtype=torch.long))
        self.register_buffer("raw_idx", torch.tensor(raw_idx, dtype=torch.long))

        m = len(proc_idx)
        self.register_buffer("lower", torch.zeros(m))
        self.register_buffer("upper", torch.zeros(m))
        self.register_buffer("mean", torch.zeros(m))
        self.register_buffer("std", torch.ones(m))

    def fit(self, X_train):

        Xp = X_train[:, self.proc_idx]
        q1 = torch.quantile(Xp, 0.25, dim=0)
        q3 = torch.quantile(Xp, 0.75, dim=0)
        iqr = q3 - q1
        self.lower.copy_(q1 - self.iqr_k * iqr)
        self.upper.copy_(q3 + self.iqr_k * iqr)

        mask_train = self._inlier_mask(X_train)
        Xp_in = X_train[mask_train][:, self.proc_idx]
        self.mean.copy_(Xp_in.mean(dim=0))
        self.std.copy_(Xp_in.std(dim=0, unbiased=False).clamp_min(self.eps))

    def _inlier_mask(self, X):
        Xp = X[:, self.proc_idx]
        return ((Xp >= self.lower) & (Xp <= self.upper)).all(dim=1)

    def forward(self, X):
        mask = self._inlier_mask(X)
        X_out = X[mask].clone()

        X_out[:, self.proc_idx] = (X_out[:, self.proc_idx] - self.mean) / self.std

        return X_out, mask


X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

pre = SimplePreprocessor(feature_names=data.feature_names)
pre.fit(X_train_t)

X_train_p, m_train = pre(X_train_t)
X_val_p, m_val = pre(X_val_t)
X_test_p, m_test = pre(X_test_t)

y_train_p = y_train_t[m_train]
y_val_p = y_val_t[m_val]
y_test_p = y_test_t[m_test]

print("Shapes finales:")
print("Train:", X_train_p.shape, y_train_p.shape)
print("Val:  ", X_val_p.shape, y_val_p.shape)
print("Test: ", X_test_p.shape, y_test_p.shape)

max_diff_latlon = (X_train_p[:, pre.raw_idx] - X_train_t[m_train][:, pre.raw_idx]).abs().max().item()
print("Max diff Latitude/Longitude:", max_diff_latlon)

Shapes finales:
Train: torch.Size([13460, 8]) torch.Size([13460])
Val:   torch.Size([1702, 8]) torch.Size([1702])
Test:  torch.Size([1698, 8]) torch.Size([1698])
Max diff Latitude/Longitude: 0.0


In [ ]:
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cpu")


class MLPRegressor(nn.Module):

    def __init__(self, num_input_features, hidden_layer_sizes, dropout=0.0, use_batch_normalization=False):
        super().__init__()
        layers = []
        previous_width = num_input_features
        for hidden_width in hidden_layer_sizes:
            layers.append(nn.Linear(previous_width, hidden_width))
            if use_batch_normalization:
                layers.append(nn.BatchNorm1d(hidden_width))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            previous_width = hidden_width
        layers.append(nn.Linear(previous_width, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_model(model, train_loader, validation_features, validation_targets, config, device):
    model = model.to(device)
    optimizer = optim.Adam(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config.get("weight_decay", 0.0),
    )
    loss_function = nn.MSELoss()
    validation_features = validation_features.to(device)
    validation_targets = validation_targets.to(device)

    training_history = {"train_loss": [], "val_loss": []}
    best_validation_loss = float("inf")
    best_model_state = None
    # Épocas seguidas sin mejorar val loss antes de parar (early stopping)
    early_stopping_epochs = config.get("early_stopping_epochs", 25)
    epochs_without_improvement = 0

    for epoch in range(config["epochs"]):
        model.train()
        running_loss_sum = 0.0
        num_samples = 0
        for batch_features, batch_targets in train_loader:
            batch_features = batch_features.to(device)
            batch_targets = batch_targets.to(device)
            optimizer.zero_grad()
            predictions = model(batch_features)
            loss = loss_function(predictions, batch_targets)
            loss.backward()
            optimizer.step()
            running_loss_sum += loss.item() * batch_features.size(0)
            num_samples += batch_features.size(0)
        train_loss = running_loss_sum / num_samples

        model.eval()
        with torch.no_grad():
            validation_predictions = model(validation_features)
            val_loss = loss_function(validation_predictions, validation_targets).item()

        training_history["train_loss"].append(train_loss)
        training_history["val_loss"].append(val_loss)

        if val_loss < best_validation_loss:
            best_validation_loss = val_loss
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= early_stopping_epochs:
                break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    model.to(device)
    return model, training_history, best_validation_loss


num_input_features = X_train_p.shape[1]

model_configs = [
    {
        "name": "Modelo 1 (4 capas ocultas, learning rate alto)",
        "hidden_layer_sizes": (64, 48, 32, 16),
        "dropout": 0.0,
        "use_batch_normalization": False,
        "batch_size": 128,
        "learning_rate": 0.01,
        "weight_decay": 0.0,
        "epochs": 200,
        "early_stopping_epochs": 30,
    },
    {
        "name": "Modelo 2 (4 capas anchas + dropout)",
        "hidden_layer_sizes": (256, 128, 96, 64),
        "dropout": 0.2,
        "use_batch_normalization": False,
        "batch_size": 64,
        "learning_rate": 0.001,
        "weight_decay": 1e-4,
        "epochs": 200,
        "early_stopping_epochs": 30,
    },
    {
        "name": "Modelo 3 (5 capas + BatchNorm)",
        "hidden_layer_sizes": (128, 128, 96, 64, 32),
        "dropout": 0.1,
        "use_batch_normalization": True,
        "batch_size": 256,
        "learning_rate": 0.003,
        "weight_decay": 1e-5,
        "epochs": 200,
        "early_stopping_epochs": 30,
    },
]

results = []
trained_models = {}

for config in model_configs:
    torch.manual_seed(42)
    train_dataset = TensorDataset(X_train_p, y_train_p)
    train_loader = DataLoader(train_dataset, batch_size=config["batch_size"], shuffle=True)

    model = MLPRegressor(
        num_input_features=num_input_features,
        hidden_layer_sizes=config["hidden_layer_sizes"],
        dropout=config["dropout"],
        use_batch_normalization=config["use_batch_normalization"],
    )
    model, training_history, best_validation_loss = train_model(
        model, train_loader, X_val_p, y_val_p, config, device
    )
    trained_models[config["name"]] = model

    model.eval()
    with torch.no_grad():
        test_predictions = model(X_test_p.to(device)).cpu()
        test_mse = nn.functional.mse_loss(test_predictions, y_test_p).item()
        test_rmse = test_mse ** 0.5

    hyperparameter_summary = (
        f"capas_ocultas={config['hidden_layer_sizes']}, "
        f"batch_size={config['batch_size']}, "
        f"learning_rate={config['learning_rate']}, "
        f"dropout={config['dropout']}, "
        f"batch_normalization={config['use_batch_normalization']}, "
        f"weight_decay={config['weight_decay']}"
    )
    results.append(
        {
            "modelo": config["name"],
            "val_mse (mejor)": round(best_validation_loss, 6),
            "test_mse": round(test_mse, 6),
            "test_rmse": round(test_rmse, 6),
            "hiperparametros": hyperparameter_summary,
        }
    )

print("Dispositivo:", device)
print("\nResumen (mejor época según val MSE):")
for row in results:
    print(row["modelo"])
    print(" ", row["hiperparametros"])
    print(f"  val MSE (mejor): {row['val_mse (mejor)']}, test MSE: {row['test_mse']}, test RMSE: {row['test_rmse']}\n")

Dispositivo: cpu

Resumen (mejor época según val MSE):
Modelo 1 (4 capas ocultas, learning rate alto)
  capas_ocultas=(64, 48, 32, 16), batch_size=128, learning_rate=0.01, dropout=0.0, batch_normalization=False, weight_decay=0.0
  val MSE (mejor): 0.379224, test MSE: 0.393696, test RMSE: 0.627452

Modelo 2 (4 capas anchas + dropout)
  capas_ocultas=(256, 128, 96, 64), batch_size=64, learning_rate=0.001, dropout=0.2, batch_normalization=False, weight_decay=0.0001
  val MSE (mejor): 0.376047, test MSE: 0.377739, test RMSE: 0.614605

Modelo 3 (5 capas + BatchNorm)
  capas_ocultas=(128, 128, 96, 64, 32), batch_size=256, learning_rate=0.003, dropout=0.1, batch_normalization=True, weight_decay=1e-05
  val MSE (mejor): 0.329099, test MSE: 0.325059, test RMSE: 0.570139



In [ ]:
import numpy as np
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    confusion_matrix,
    classification_report,
)

selected_model_name = "Modelo 1 (4 capas ocultas, learning rate alto)"
selected_model = trained_models[selected_model_name]

selected_model.eval()
with torch.no_grad():
    y_pred_test = selected_model(X_test_p.to(device)).cpu().numpy()

y_true_test = y_test_p.cpu().numpy()

mae_value = mean_absolute_error(y_true_test, y_pred_test)
mse_value = mean_squared_error(y_true_test, y_pred_test)
rmse_value = mse_value ** 0.5
r2_value = r2_score(y_true_test, y_pred_test)

print(f"Evaluación de regresión para {selected_model_name}:")
print(f"MAE:  {mae_value:.6f}")
print(f"MSE:  {mse_value:.6f}")
print(f"RMSE: {rmse_value:.6f}")
print(f"R2:   {r2_value:.6f}")

price_threshold = float(np.median(y_true_test))
y_true_binary = (y_true_test >= price_threshold).astype(int)
y_pred_binary = (y_pred_test >= price_threshold).astype(int)

print(f"\nUmbral binario usado (mediana del precio): {price_threshold:.6f}")
print(f"\nConfusion Matrix for {selected_model_name}:")
print(confusion_matrix(y_true_binary, y_pred_binary))

print(f"\nClassification Report for {selected_model_name}:")
print(classification_report(y_true_binary, y_pred_binary))

Evaluación de regresión para Modelo 1 (4 capas ocultas, learning rate alto):
MAE:  0.450794
MSE:  0.393696
RMSE: 0.627452
R2:   0.650280

Umbral binario usado (mediana del precio): 1.804500

Confusion Matrix for Modelo 1 (4 capas ocultas, learning rate alto):
[[677 172]
 [125 724]]

Classification Report for Modelo 1 (4 capas ocultas, learning rate alto):
              precision    recall  f1-score   support

           0       0.84      0.80      0.82       849
           1       0.81      0.85      0.83       849

    accuracy                           0.83      1698
   macro avg       0.83      0.83      0.82      1698
weighted avg       0.83      0.83      0.82      1698

